# V3：三类扩展联合模型

比较 static_stacking 与 combined_stacking，分别使用静态特征和静态+轨迹+既往ICD表示。

联合模型每个基模型的训练 Pipeline 都重新拟合自身的轨迹筛选、插补、LASSO、TF-IDF、SVD及缩放，stacking使用3折交叉拟合，基模型不重复调参。不能先在完整开发集生成 embedding 再堆叠。

这是探索性的联合消融。必须先看独立时间/embedding实验，再判断联合模型是否改善AUROC、AUPRC、Brier、校准和DCA。当前不使用已看过的固定留出集选择最优扩展，也没有外部验证结论。

**balanced 简化计算版**：固定基模型参数、full保留5×10外层验证；LASSO内部5折/30λ，stacking内部3折。与上级严格版使用独立缓存；前6折旧预测不混入。当前没有运行新版本真实患者模型。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 从扩展目录或 notebooks 子目录定位项目，避免硬编码 Windows 盘符。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从 dn_mortality_repro_V3_extensions 或其 notebooks 子目录启动。")
sys.path.insert(0, str(ROOT))
print("扩展目录：", ROOT)


In [ ]:
# pilot 是1×3外层小规模试跑；正式实验将 PROFILE 改为 "full"。
# 仅把 RUN_EXPERIMENT 改成 True 才会启动；full可能需要较长时间。
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_stacking', 'combined_stacking']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("尚未启动本Notebook中的模型训练。")


In [ ]:
# 优先查看正式 full 结果；没有结果时打印状态，不重新训练。
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("详细记录：", folder / "report.md")
else:
    print("该模式尚无实验结果。pilot 结果需把 RESULT_PROFILE 改为 pilot 才显示。")


## 实验状态与结论（自动更新）

运行模式：`full`；方法：balanced 固定参数；结果目录：`results/full/27464ba98305edbc`。

LASSO 5折/30λ、stacking 3折；静态与扩展使用同一简化方法。严格版的部分折没有混入。

固定留出集未参与本实验。树模型扩展不需要 stacking 内部交叉拟合；原固定参数及训练折内预处理不变。

Holm 校正覆盖当前已有配对比较；后续增加实验时，校正 p 值可能变化。

```text

       experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
combined_stacking             50          0.80475        0.02738          0.61572          0.15038                      1.08695    27443.74099            0.80347            0.60377            0.15038                   -0.00846                1.05953
  static_stacking             50          0.80466        0.02657          0.61586          0.15048                      1.07477     1319.71249            0.80325            0.60386            0.15047                    0.00134                1.05015

```

配对差异：

```text

       experiment       reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
  static_stacking       static_lr  AUROC    0.00539  0.00823       0.00298 -0.00060 0.01138  1.80933  49  0.07654 10  5                    1.0
  static_stacking       static_lr  AUPRC    0.00847  0.01519       0.00550 -0.00259 0.01952  1.53947  49  0.13012 10  5                    1.0
  static_stacking       static_lr  Brier   -0.00044  0.00289       0.00105 -0.00254 0.00166 -0.41929  49  0.67683 10  5                    1.0
combined_stacking static_stacking  AUROC    0.00009  0.01052       0.00381 -0.00757 0.00774  0.02245  49  0.98218 10  5                    1.0
combined_stacking static_stacking  AUPRC   -0.00015  0.02065       0.00748 -0.01517 0.01487 -0.01986  49  0.98424 10  5                    1.0
combined_stacking static_stacking  Brier   -0.00010  0.00393       0.00142 -0.00296 0.00276 -0.06938  49  0.94497 10  5                    1.0

```

- `static_stacking` 相对 `static_lr`：ΔAUROC=0.0054，校正95% CI [-0.0006, 0.0114]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `combined_stacking` 相对 `static_stacking`：ΔAUROC=0.0001，校正95% CI [-0.0076, 0.0077]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

> 历史快照：第五次重复现已完成；请以本 Notebook 后面的“Stacking 最终开发集消融记录”为准。

# 联合扩展消融：前四次重复的会议记录（2026-10-14，暂定）

**目前结论：联合扩展没有显示出对静态 stacking 的额外性能提升。**

这是一份中途查看结果的快照，不是最终报告。仅使用完整的 rep 1–4（seed 42、43、44、45），每次 10 折，共 40 折；rep 5 的所有折均排除。所有模型都截取相同的前四次重复，并核对每折验证患者索引一致。开发集 2,045 人、死亡 544 人；没有读取固定留出集标签，没有重新训练或修改运行中的代码与缓存。

## 同一批 40 折的比较

| 模型 | AUROC（40折均值） | AUPRC（40折均值） | Brier（40折均值，低更好） |
|---|---:|---:|---:|
| 静态 LR | 0.7986 | 0.6073 | 0.1512 |
| 静态 RF | 0.7972 | 0.6051 | 0.1520 |
| 静态 XGBoost | 0.7946 | 0.6041 | 0.1527 |
| 静态 stacking | 0.8038 | 0.6145 | 0.1508 |
| 静态 + 时间特征 LR | 0.7942 | 0.5992 | 0.1533 |
| 静态 + 既往诊断负担 LR | 0.7985 | 0.6064 | 0.1512 |
| 静态 + 既往诊断负担 + ICD embedding LR | 0.7954 | 0.6044 | 0.1527 |
| 静态 + 时间特征 + 既往诊断表示 stacking | 0.8038 | 0.6129 | 0.1510 |

以上每折先计算指标，再对 40 折取平均。每次重复中把 10 折的 OOF 概率拼回完整开发集，再算指标，最后对四次重复取平均，会得到另一组数字：

| 模型 | AUROC | AUPRC | Brier |
|---|---:|---:|---:|
| 静态 stacking | 0.802404 | 0.602385 | 0.150758 |
| 静态 + 时间特征 + 既往诊断表示 stacking | 0.802430 | 0.600443 | 0.150977 |

两种汇总方式回答的比较略有不同，不能混用。联合模型的 pooled AUROC 仅高 0.000026；40 折平均 AUROC 反而低 0.000067。两者都表明 AUROC 实际上几乎没有变化。这里的均值来自重复交叉验证，不是单次训练，也不是 95% 置信区间。

## 联合模型是否稳定优于静态 stacking？

| 重复（seed） | 静态 stacking AUROC | 联合 stacking AUROC | 联合减静态 |
|---|---:|---:|---:|
| 42 | 0.803914 | 0.802084 | -0.001830 |
| 43 | 0.800608 | 0.800264 | -0.000344 |
| 44 | 0.800782 | 0.801098 | +0.000316 |
| 45 | 0.804311 | 0.806275 | +0.001964 |

四次重复中，pooled AUROC 两次上升、两次下降；40 个配对折中，25 折上升、15 折下降。折的大小、差值大小和重复之间的依赖都要考虑，因此不能只按“赢了多少折”判断显著提升。

Nadeau–Bengio 校正重复交叉验证的依赖后，联合模型减静态 stacking：

- AUROC 平均差值 −0.000067，95% 差值区间 [−0.007817, +0.007683]，p=0.9861。
- AUPRC 平均差值 −0.001629，95% 差值区间 [−0.016767, +0.013509]，p=0.8288。
- Brier 平均差值 +0.000216，95% 差值区间 [−0.002511, +0.002944]，p=0.8734；正值表示略差。
- 对当前 7 个比较分别按指标做 Holm 校正，这三个比较的校正 p 值均为 1。

这是探索性的中途区间和检验，不能作为预先规定的最终确认性检验。没有检测到差异也不能证明两模型等效。保持原定第五次重复，最终完整结果另行汇总。

## 校准、临床效用与特征是否真正进入模型

四次 pooled 校准斜率均值：静态 stacking 1.0470，联合 stacking 1.0532，接近且都在 1 附近；Brier 未改善。0.05–0.60 的 56 个 DCA 阈值点中，联合模型平均净获益在 32 点较高、22 点较低、2 点相同；差值很小且未计算不确定性，不能声称整体临床效用改善，也不能将不同阈值当成独立样本。

检查联合模型 40 个外层最终拟合的 LR 基模型：每折都选中了至少 1 个时间特征，中位数 4 个（范围 1–13）；每折都有 32 维 ICD embedding。既往诊断负担的 4 个数值指标在 LASSO 筛选之后直接加入，并未被排除。时间特征参与 LASSO；诊断负担和 embedding 直接追加，因此 selected_numeric 中没有诊断负担列不代表模型没用它们。32 维 embedding 存在也不等于每维均有有效贡献。联合模型每折由 LASSO 选中的数值变量中位数为 24，此外还追加 4 个诊断负担指标及 32 维 embedding。

## 目前怎么解释

静态 stacking 相对静态 LR 的 40 折 AUROC 均值提高约 0.0052，但校正区间仍包含 0（−0.001436 至 +0.011871，未作多重校正 p=0.1208）。可把它作为有潜力的候选模型，不能说已经统计学确认优于 LR。

时间轨迹、既往诊断负担和 ICD embedding 已完成实验，但当前实现没有体现明确额外增益。可能原因包括：静态特征和现有评分已捕捉部分病情信息；24 小时内部分化验重复测量有限；既往诊断记录与院外完整病史不等价；增加维度可能增加估计波动。这些是待验证的解释，不是本次比较已经证明的原因。结果仅适用于当前特征构建、选择和固定参数版本，不能推断所有时间模型或多模态方法都无效。这里的 embedding 来自结构化 ICD 诊断，不是临床文本。

本次前四次联合模型各折记录用时合计约 6.33 小时，静态 stacking 约 17.94 分钟。缓存复用和并行负载不同，这不是严格受控的耗时实验，但联合流程的计算代价明显较高。现阶段可优先保留静态 stacking 和 LR 对照，等第五次重复完成后判断最终取舍。

## 会上可以这样说

> 我们在同一开发集、相同划分上做了 stacking、时间轨迹和既往诊断表示的消融。前四次十折重复里，静态 stacking 的平均 AUROC 约 0.804，联合模型仍约 0.804，AUPRC 和 Brier 也没有改善，配对差值区间包含 0。因此目前比较有希望的是静态 stacking，时间和诊断扩展暂时没有带来额外收益。第五次重复还在跑，最终结果和独立验证要分开报告。不能直接拿这里的开发集 CV 数字和原文的 0.780 判断谁更好，因为队列与评估设置不相同。

完整训练不会因为本次中途查看而调整。论文表述应如实呈现消融结果，后续可围绕选定候选模型完成校准、解释性和临床效用评估，不再以反复观察同一开发集来保证 AUC 上升。


# Stacking 最终开发集消融记录（2026-10-14）

已完成原定 5 次重复 × 10 折，共 50 折，seed 42–46。开发集 2,045 人、死亡 544 人。同一验证折作配对比较，未读取固定留出集标签，没有额外训练。这份完整记录取代前四次重复的暂定结论；中途快照保留用于追踪。

**结论：静态 stacking 相对当前最佳单模型 LR 有小幅数值提升，五次重复方向一致，但未达到统计显著。联合加入时间轨迹和既往诊断表示后，没有显示额外收益。**

| 模型 | AUROC（50折均值） | AUPRC（50折均值） | Brier（50折均值，低更好） |
|---|---:|---:|---:|
| 静态 LR | 0.799275 | 0.607397 | 0.150918 |
| 静态 RF | 0.797980 | 0.606182 | 0.151771 |
| 静态 XGBoost | 0.795041 | 0.605724 | 0.152496 |
| 静态 stacking | 0.804665 | 0.615863 | 0.150479 |
| 联合扩展 stacking | 0.804750 | 0.615715 | 0.150381 |

这些是先在每个外层验证折算指标，再对 50 折求平均。另一种“每次重复拼回完整开发集 OOF 概率后算指标，再对五次取平均”的 AUROC 分别为：静态 LR 0.798852、静态 stacking 0.803249、联合 stacking 0.803473。两种汇总方式不要混用。

## 静态 stacking 相对单模型

相对静态 LR，50 折平均 AUROC 增加 0.005389（约 0.54 个百分点），AUPRC 增加 0.008466，Brier 减少 0.000439。静态 stacking 的平均 AUROC 和 AUPRC 也高于当前固定参数的 RF、XGBoost；没有因此推断它优于所有调参后的单模型。

五次重复的 pooled OOF AUROC 和 AUPRC 均高于 LR；50 个折中 41 折的 AUROC 更高。重复使用同一开发集，这些并非五个独立外部验证样本。

Nadeau–Bengio 校正后的 AUROC 配对差值 95% 区间为 [−0.000596, +0.011375]，p=0.076535；对当前 7 个 AUROC 比较作 Holm 校正后 p=0.535748。AUPRC 差值区间 [−0.002585, +0.019518]，p=0.130124；Brier 差值区间 [−0.002541, +0.001663]，p=0.676835。因此可描述为“数值上有小幅提升，重复划分方向一致”，不能写成“显著优于 LR”。这里的差值区间不是 stacking 模型自身 AUROC 的置信区间。

## 联合扩展是否进一步提升 stacking

联合模型相对静态 stacking 的 50 折平均 AUROC 只增加 0.000086（约 0.009 个百分点），AUPRC 减少 0.000148，Brier 减少 0.000099，实际数值非常接近。

AUROC 配对差值 95% 区间为 [−0.007569, +0.007740]，p=0.982179，Holm p=1。五次重复的 pooled AUROC 三次上升、两次下降；50 折中 30 折上升。不能只凭赢的折数证明提升，也不能用未显著来证明严格等效。

| seed | 静态 stacking − LR AUROC | 联合 − 静态 stacking AUROC |
|---|---:|---:|
| 42 | +0.003559 | -0.001830 |
| 43 | +0.005730 | -0.000344 |
| 44 | +0.004561 | +0.000316 |
| 45 | +0.004749 | +0.001964 |
| 46 | +0.003383 | +0.001015 |

## 报告与后续实验的边界

当前可以将静态 stacking 作为候选模型，保留 LR 作为简单且有竞争力的对照。对联合模型，报告“当前扩展未带来明确额外增益”，无需为接近零的平均差值宣称改进。数值提升可能来自基模型预测互补，但本次指标比较没有直接验证提升机制。

本轮为固定基模型参数、简化内层计算的 balanced 版本，属于开发集探索性结果。不能与原文 0.780 直接比较并宣称超越原文，队列和评估方式不同。固定留出集若已看过，也不能当作全新未触碰测试集选择扩展。

独立的时间、诊断负担和 embedding 消融目前只评估了 LR。联合 stacking 的 LR、RF、XGBoost 均输入了联合扩展，但尚未单独报告 RF、XGBoost 的新增特征成绩。因此“联合没有收益”不能推广为“时间特征或 embedding 对每种模型都无效”。四个诊断负担指标与 ICD embedding 是在数值 LASSO 筛选后直接追加，未被 LASSO 排除；时间特征参与数值筛选。

会上可以说：

> 完整五次十折重复里，静态 stacking 的平均 AUROC 从 LR 的 0.799 提高到 0.805，五次重复均更高，但配对差值区间仍包含零，提升尚未得到统计学确认。加入时间轨迹和诊断表示后仍约 0.805，没有额外增益。目前静态 stacking 是一个有希望的候选，LR 仍是重要对照。
